# 用「凍結的 Embedding 模型 + 邏輯回歸」做影評情緒分類

---

## 這份程式要做什麼

判斷一句影評是**正面**還是**負面**，這是你之前用「微調 BERT」做過的同一個任務。這次換一種做法：

```
影評文字
   │  ① Embedding 模型（all-mpnet-base-v2，凍結不動）
   ▼
一個 768 維的向量（代表這句話的語意）
   │  ② 邏輯回歸（只訓練這個小分類器）
   ▼
正面 / 負面
```

## 核心想法：特徵萃取（Feature Extraction）

| 部分 | 角色 | 是否訓練 |
|---|---|---|
| Embedding 模型 | **把文字變成向量**（萃取特徵） | **否，完全凍結** |
| 邏輯回歸 | **看向量，決定正面或負面** | **是，只訓練它** |

Embedding 模型已經在海量資料上學會「語意相近的句子，向量也相近」，我們**直接拿它的成果來用**，只需要再訓練一個很小的分類器。

## 這和「微調」差在哪

你之前做過的微調，是把整個 BERT（約 1 億個參數）拿來更新。這一份完全不更新 Embedding 模型：

| | **微調 BERT（之前）** | **凍結 Embedding + 邏輯回歸（這份）** |
|---|---|---|
| 更新的參數 | 整個模型（約 1 億） | **只有分類器（約 769 個）** |
| 需要 GPU 訓練 | 通常需要 | **不需要**（只有編碼那步用 GPU 會更快） |
| 訓練時間 | 較長 | **很短**（訓練分類器只要幾秒） |
| 實作複雜度 | 較高（Trainer、參數、標籤） | **很低** |
| 對任務的適應程度 | 高（模型針對任務調整） | 較低（向量是通用的，沒針對這個任務調整） |
| 效果 | 通常較好 | 通常稍低，但已經相當不錯 |

這是一個典型的取捨：**簡單快速** vs **效果最佳**。實務上常常先用這種做法建立一個基準線（baseline），再決定要不要投入更多資源微調。

## 流程總覽

```
載入資料 → 載入 Embedding 模型 → 把訓練集與測試集都編碼成向量
   → 用訓練集的向量訓練邏輯回歸 → 預測測試集 → 用分類報告評估
```

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `load_dataset` | 載入影評資料集 |
| `SentenceTransformer` | 載入 Embedding 模型，把文字轉成向量 |
| `LogisticRegression`（scikit-learn） | 邏輯回歸分類器 |
| `classification_report`（scikit-learn） | 產生分類評估報告（precision、recall、F1 等） |

**scikit-learn（`sklearn`）** 是 Python 最常用的傳統機器學習套件。這份程式把「深度學習的 Embedding 模型」和「傳統機器學習的分類器」組合在一起使用。


In [ ]:
#====Embedding Model====
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

## 步驟 1：定義評估函式

```python
def evaluate_performance(y_true, y_pred):
    report = classification_report(
        y_true, y_pred, target_names=["negative", "positive"])
    print(report)
```

| 部分 | 說明 |
|---|---|
| `y_true` | **真正的答案**（資料集裡的標籤） |
| `y_pred` | **模型的預測** |
| `target_names=["negative", "positive"]` | 把標籤的數字換成看得懂的名稱 |

### `target_names` 的順序很重要

標籤編號是 **0 = 負面、1 = 正面**，`target_names` 的順序必須和編號對應：第 0 個是 `"negative"`、第 1 個是 `"positive"`。順序寫反，報告裡「負面」和「正面」的數字就會互換，而且不會報錯。

### 把評估包成函式的好處

之後想評估別的分類器（換個模型、換個參數），只要再呼叫一次 `evaluate_performance`，不用重寫。

函式裡**只 print、沒有回傳**。需要把結果存起來時，可以改成 `return report`。

In [ ]:
def evaluate_performance(y_true, y_pred):
    report = classification_report(
        y_true, y_pred, target_names=["negative", "positive"])
    print(report)

## 步驟 2：載入資料

```python
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
```

Rotten Tomatoes 是影評資料集。每筆資料有：

| 欄位 | 說明 |
|---|---|
| `text` | 一句影評 |
| `label` | `0` = 負面、`1` = 正面 |

資料已經切成 `train`（約 8530 筆）、`validation`（約 1066 筆）、`test`（約 1066 筆）

```python
# print(data)                  # 看有哪幾份資料、各有幾筆
# print(data["train"][0])      # 印出第一筆
```

In [ ]:
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
# print(data)
# print(data["train"][0])  #印出第一筆資料

## 步驟 3：載入 Embedding 模型

```python
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
```

`all-mpnet-base-v2` 是一個**已經訓練好的句向量模型**：丟一句話進去，吐出一個 **768 維**的向量。

| 特性 | 說明 |
|---|---|
| 向量維度 | 768 |
| 訓練方式 | 在大量句子對上用對比學習訓練：語意相近的句子，向量靠近 |
| 這份程式怎麼用它 | **只拿來編碼，不訓練**（凍結） |

如果有 GPU，`SentenceTransformer` 會自動使用，編碼會快很多。

### 為什麼選這個模型

你在微調 Embedding 的幾份程式裡已經看過它：它是從 `all-mpnet-base-v2` 這類現成模型出發，效果通常比自己用 `bert-base` 從零訓練的句向量好很多。

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

## 步驟 4：把文字編碼成向量

```python
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)
```

| 部分 | 說明 |
|---|---|
| `data["train"]["text"]` | 訓練集的所有影評（一個清單） |
| `model.encode(...)` | 把清單裡**每一句**都轉成向量 |
| `show_progress_bar=True` | 顯示進度條 |

### 結果的樣子

`encode` 回傳一個 **numpy 陣列**：

```
train_embeddings 的形狀：(訓練筆數, 768)     例如 (8530, 768)
test_embeddings  的形狀：(測試筆數, 768)     例如 (1066, 768)
```

每一列是一句影評的 768 維向量。**這個陣列就是後面分類器的輸入特徵。**

### 重要觀念

- **編碼時完全沒有用到標籤**：`encode` 只看文字，所以不會有「偷看答案」的問題
- **訓練集和測試集用同一個模型編碼**：兩邊的向量才在同一個空間裡，分類器才能把訓練集學到的規則用在測試集上
- 這一步通常是整份程式**最慢**的部分（要跑完所有句子），要等一下

### 記憶體

8530 × 768 個 32 位元浮點數，約 26 MB，很小。

In [ ]:
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

## 步驟 5：訓練分類器（邏輯回歸）

```python
clf = LogisticRegression(random_state = 42)
clf.fit(train_embeddings, data["train"]["label"])
```

### 什麼是邏輯回歸

名稱有「回歸」，但它其實是**分類器**。它是一個**線性模型**：

```
分數 = w1×x1 + w2×x2 + ... + w768×x768 + b          ← 對 768 個向量數字加權求和
機率 = 1 / (1 + e^(−分數))                          ← 用 sigmoid 轉成 0~1 的機率
機率 > 0.5 → 預測「正面」，否則預測「負面」
```

**訓練 = 找出最好的權重 `w` 和偏差 `b`。**

### `fit` 在做什麼

| 部分 | 說明 |
|---|---|
| `train_embeddings` | 輸入特徵，形狀 `(筆數, 768)` |
| `data["train"]["label"]` | 答案（0 或 1） |
| `clf.fit(X, y)` | 找出能讓預測盡量符合答案的權重 |

邏輯回歸對這種大小的資料，**通常幾秒到幾十秒**就訓練完（我沒有實測你的資料）。

### `random_state = 42`

固定隨機種子，讓結果可重現。我印象中預設的求解器（lbfgs）本身是確定性的，這個參數主要對某些其他求解器才有影響，不過加上它是好習慣，沒有害處。

### 可能出現的警告

預設的迭代次數上限（`max_iter`）是 100。768 維的特徵有時會在 100 次內還沒完全收斂，終端機可能出現 `ConvergenceWarning`。這通常不影響結果的可用性，如果想消除，可以把 `max_iter` 調大（例如 1000）。我沒有確認你這份資料是否會出現。

In [ ]:
clf = LogisticRegression(random_state = 42)
clf.fit(train_embeddings, data["train"]["label"])

## 步驟 6：預測測試集

```python
y_pred = clf.predict(test_embeddings)
```

用訓練好的分類器，對**測試集的向量**做預測。`y_pred` 是一個 numpy 陣列，每個元素是 0 或 1：

In [ ]:
y_pred = clf.predict(test_embeddings)

## 步驟 7：評估

```python
evaluate_performance(data["test"]["label"], y_pred)
```

**注意參數順序：第一個是真正的答案，第二個是預測。** 傳反的話，precision 和 recall 會互換。

### 怎麼讀分類報告

以下是我用縮小版合成資料實測產生的報告（數字只是示範格式，不是這份影評資料的結果）：

```
              precision    recall  f1-score   support

    negative       0.94      0.94      0.94       200
    positive       0.94      0.94      0.94       200

    accuracy                           0.94       400
   macro avg       0.94      0.94      0.94       400
weighted avg       0.94      0.94      0.94       400
```

| 欄位 | 意思 |
|---|---|
| `precision`（精確率） | 模型**說是這一類**的，有多少**真的是**這一類 |
| `recall`（召回率） | **真正屬於這一類**的，模型**找出了多少** |
| `f1-score` | precision 和 recall 的調和平均，兩者都高才會高 |
| `support` | 測試集裡**真正屬於這一類**的筆數 |
| `accuracy` | 整體答對的比例 |
| `macro avg` | 兩類各自的分數**直接平均**（每類一樣重要） |
| `weighted avg` | 兩類各自的分數依 `support` **加權平均** |

### 這份資料的結果

Rotten Tomatoes 的測試集正負各半，所以兩類的 `support` 大約相同，`macro avg` 和 `weighted avg` 會很接近，看 `accuracy` 或 `f1-score` 就夠了。

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

---

# 重點整理

## 一、整份程式一句話

> 用現成的 Embedding 模型（凍結不動）把每句影評編碼成 768 維向量，只用這些向量訓練一個邏輯回歸分類器，再對測試集預測，並用分類報告評估正面／負面的判斷表現。

## 二、必須搞懂的 10 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | 特徵萃取 | 用現成模型把原始資料變成向量，再丟給簡單的分類器 |
| 2 | 凍結（frozen） | Embedding 模型的權重完全不更新 |
| 3 | `encode` | 把文字清單轉成形狀 `(筆數, 維度)` 的向量陣列 |
| 4 | 邏輯回歸 | 線性分類器，用加權求和加 sigmoid 輸出機率 |
| 5 | 可訓練參數 | 維度 + 1（768 + 1 = 769 個），遠少於 BERT 微調 |
| 6 | `fit` / `predict` | scikit-learn 通用介面：訓練、預測 |
| 7 | precision | 模型說是這類的，有多少真的是 |
| 8 | recall | 真正是這類的，模型找出了多少 |
| 9 | F1 | precision 與 recall 的調和平均 |
| 10 | baseline（基準線） | 先用最簡單的做法建立一個參考分數 |

## 三、程式流程速記

```
1. 載入 load_dataset、SentenceTransformer、LogisticRegression、classification_report
2. 定義 evaluate_performance(y_true, y_pred)
3. 載入 Rotten Tomatoes（train / test）
4. 載入 all-mpnet-base-v2（凍結）
5. encode 訓練集與測試集 → 兩個 (筆數, 768) 的陣列
6. LogisticRegression.fit(訓練向量, 訓練標籤)
7. predict(測試向量) → y_pred
8. evaluate_performance(測試標籤, y_pred)
```

## 四、最容易搞混的地方

1. **這份有「訓練」嗎？**
   有，但只訓練邏輯回歸（約 769 個參數）。Embedding 模型沒有訓練。

2. **為什麼測試集也要用同一個模型編碼？**
   兩邊的向量要在同一個空間裡，分類器學到的規則才能套用到測試集。

3. **編碼時為什麼不用標籤？**
   `encode` 只看文字，標籤只用在訓練分類器。

4. **邏輯回歸是回歸還是分類？**
   名字有「回歸」，但它是分類器。

5. **`y_true` 和 `y_pred` 順序要注意嗎？**
   要。傳反了，precision 和 recall 會互換。

6. **`target_names` 順序為什麼重要？**
   必須和標籤編號對應（0 = negative、1 = positive），寫反不會報錯，但報告的名稱會對調。

7. **accuracy 和 F1 看哪個？**
   測試集正負各半時兩者通常很接近，看哪個都可以；類別不平衡時，F1 更有參考價值。

8. **為什麼這個做法比微調省資源？**
   不需要反向傳播更新大模型，只需要做一次前向編碼，加上訓練一個小分類器。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| `ConvergenceWarning` | 邏輯回歸在 `max_iter` 內沒完全收斂 | 調大 `max_iter`，例如 `LogisticRegression(max_iter=1000)` |
| 編碼非常慢 | 在 CPU 上跑，或資料量大 | 使用 GPU；或先用少量資料測試 |
| `encode` 報型別錯誤 | 傳入的不是一般的 list（新版 `datasets` 取欄位可能得到特殊物件） | 轉成 list：`list(data["train"]["text"])` |
| 報告裡 negative 和 positive 的分數顛倒 | `target_names` 順序和標籤編號不一致 | 確認 0 = negative、1 = positive |
| precision 與 recall 看起來互換 | `y_true` 與 `y_pred` 傳反了 | 第一個是真正答案，第二個是預測 |
| 載入模型很慢 | 第一次執行，正在下載 | 等待，之後從快取讀取 |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **每次執行都重新編碼**
   編碼最慢，結果卻沒有存起來。可以用 `np.save` 存下向量，之後直接讀。

2. **沒有使用驗證集**
   只用 train 和 test。沒有調整超參數，所以直接看 test 是合理的；如果要調 `C`、換分類器，應該用 validation 集，test 留到最後。

3. **分類器和向量都沒有儲存**
   跑完就消失了。

4. **邏輯回歸用預設參數**
   沒有調整正則化強度 `C`，也沒有調 `max_iter`。

5. **`evaluate_performance` 只 print 不回傳**
   想把數字存起來比較，需要改成 `return`。

6. **Embedding 沒有針對任務調整**
   向量是通用的，沒有針對「情緒」這個任務最佳化，這是這種做法效果略低於微調的主因。

7. **只有單次結果**
   沒有做交叉驗證，無法知道分數的波動。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 看向量的形狀
print(train_embeddings.shape)                 # (訓練筆數, 768)
print(test_embeddings.shape)
print(train_embeddings[0][:5])                # 第一句話向量的前五個數字

# 2. 存下向量，之後不用重新編碼
import numpy as np
np.save("train_embeddings.npy", train_embeddings)
np.save("test_embeddings.npy", test_embeddings)

# 3. 消除收斂警告
clf = LogisticRegression(random_state=42, max_iter=1000)

# 4. 換成別的分類器，比較結果
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
for c in (LinearSVC(), KNeighborsClassifier(n_neighbors=15)):
    c.fit(train_embeddings, data["train"]["label"])
    print(type(c).__name__)
    evaluate_performance(data["test"]["label"], c.predict(test_embeddings))

# 5. 看分類器對單句的預測機率
probs = clf.predict_proba(test_embeddings[:5])
print(probs)                                  # 每列是 [負面機率, 正面機率]

# 6. 看看分錯的句子，找出模型的弱點
import numpy as np
wrong = np.where(y_pred != np.array(data["test"]["label"]))[0]
for i in wrong[:5]:
    print(data["test"]["label"][i], y_pred[i], data["test"]["text"][i])

# 7. 零樣本分類（完全不訓練分類器）：比較句向量與「標籤描述」的相似度
from sentence_transformers import util
label_emb = model.encode(["A negative review", "A positive review"])
sims = util.cos_sim(test_embeddings, label_emb)
zero_shot_pred = sims.argmax(dim=1).numpy()
evaluate_performance(data["test"]["label"], zero_shot_pred)

# 8. 用交叉驗證看分數的穩定度
from sklearn.model_selection import cross_val_score
scores = cross_val_score(LogisticRegression(max_iter=1000),
                         train_embeddings, data["train"]["label"], cv=5, scoring="f1")
print(scores, scores.mean())

# 9. 和你之前微調 BERT 的 F1 做比較
```

## 八、與前面學過的分類方法對照

| | 微調 BERT | SetFit | **凍結 Embedding + 邏輯回歸（這份）** | 零樣本（標籤描述） |
|---|---|---|---|---|
| 需要標註資料 | 多（幾千筆） | 極少（幾十筆） | 中等（幾千筆較好） | **不需要** |
| 更新 Embedding 模型 | 是 | 是 | **否** | 否 |
| 訓練的參數 | 約 1 億 | Embedding 微調 + 分類器 | **約 769** | 無 |
| 訓練成本 | 高 | 中 | **低** | **無** |
| 實作複雜度 | 高 | 中 | **低** | 最低 |
| 效果（一般） | 最好 | 好 | 不錯 | 較低 |

這幾種方法**都以預訓練模型為起點**，差別只在於「更新多少、需要多少標註資料」。

## 九、怎麼選用哪一種

| 你的情況 | 建議 |
|---|---|
| 想快速建立基準線、資源有限 | **這份：凍結 Embedding + 邏輯回歸** |
| 標註資料非常少（幾十筆） | SetFit |
| 標註資料充足、追求最佳效果 | 微調整個模型 |
| 完全沒有標註 | 零樣本（標籤描述）或用生成式模型 |

## 十、一句話記住

> **凍結的 Embedding 模型負責把文字變成向量，邏輯回歸負責在向量上畫一條分界線。** 只訓練約 769 個參數，就能在影評情緒分類上得到不錯的基準線。